# 🔬 DermaScan — Model Training Notebook

**Dataset:** HAM10000 — 10,015 dermoscopic images across 7 skin condition classes  
**Model:** EfficientNetB0 (ImageNet pre-trained) fine-tuned via transfer learning  
**Target:** ~85%+ validation accuracy

---
### ⚡ Quick start
1. Make sure **Runtime → Change runtime type → T4 GPU** is selected
2. Run all cells top to bottom (Runtime → Run all)
3. At the end, `dermascan.keras` will download automatically
4. Place that file in `backend/model/dermascan.keras`

## Section 1 — Install & Kaggle Setup

In [ ]:
# Install dependencies
!pip install -q kaggle scikit-learn matplotlib seaborn
print('✅ Dependencies installed')

In [ ]:
# Upload your kaggle.json API token
# Get it from: kaggle.com → Profile → Settings → API → Create New Token
from google.colab import files
print('📂 Please upload your kaggle.json file when prompted...')
uploaded = files.upload()

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials configured')

## Section 2 — Download HAM10000 Dataset

In [ ]:
# Download HAM10000 (~1.5 GB, takes ~2-4 min on Colab)
!kaggle datasets download -d kmader/skin-lesion-analysis-toward-melanoma-detection \
    --unzip -p /content/ham10000 -q
print('✅ Dataset downloaded')
!ls /content/ham10000

## Section 3 — Data Exploration & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import os, glob
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Load metadata CSV
meta_path = '/content/ham10000/HAM10000_metadata.csv'
df = pd.read_csv(meta_path)
print(f'Total samples: {len(df)}')
print(df['dx'].value_counts())

# Class distribution
plt.figure(figsize=(10, 4))
df['dx'].value_counts().plot(kind='bar', color='teal', edgecolor='white')
plt.title('HAM10000 Class Distribution', fontsize=14)
plt.xlabel('Condition Code')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Build image path lookup (images span two folders)
all_image_paths = glob.glob('/content/ham10000/**/*.jpg', recursive=True)
img_id_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in all_image_paths}
print(f'Total images found: {len(img_id_to_path)}')

# Map image_id → file path
df['path'] = df['image_id'].map(img_id_to_path)
df = df.dropna(subset=['path'])
print(f'Matched samples: {len(df)}')

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['dx'])
CLASS_NAMES = list(le.classes_)
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

# Stratified train/val split
train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}')

# Compute class weights to handle imbalance
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_df['label'].values
)
class_weights = dict(enumerate(weights))
print('Class weights:', {CLASS_NAMES[k]: round(v, 2) for k, v in class_weights.items()})

## Section 4 — Build Data Generators

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    img = tf.image.rot90(img, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))
    return img, label

def preprocess_efficientnet(img, label):
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img, label

def make_dataset(dataframe, augment_data=False):
    paths = dataframe['path'].values
    labels = dataframe['label'].values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(preprocess_efficientnet, num_parallel_calls=AUTOTUNE)
    if augment_data:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, augment_data=True)
val_ds   = make_dataset(val_df,   augment_data=False)
print('✅ Datasets ready')

## Section 5 — Build & Train EfficientNetB0

In [ ]:
from tensorflow.keras import layers, Model

def build_model(num_classes=7):
    base = tf.keras.applications.EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False  # Phase 1: frozen base

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs), base

model, base_model = build_model(NUM_CLASSES)
model.summary()

In [ ]:
import os

os.makedirs('/content/checkpoints', exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/checkpoints/best_phase1.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

# ── Phase 1: Train head only (base frozen) ────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('\n🚀 Phase 1: Training classification head (5 epochs)...')
h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    class_weight=class_weights,
    callbacks=callbacks
)

In [ ]:
# ── Phase 2: Fine-tune top layers of base ─────────────────────────────────────
base_model.trainable = True

# Freeze all but the top 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/checkpoints/best_phase2.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
]

print('\n🚀 Phase 2: Fine-tuning top 30 layers (15 epochs)...')
h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_p2
)

## Section 6 — Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Collect all val predictions
y_true, y_pred = [], []
for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print('\n📊 Classification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — DermaScan EfficientNetB0', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Section 7 — Save & Download Model

In [ ]:
# Save model in Keras format
model.save('/content/dermascan.keras')
print('✅ Model saved to /content/dermascan.keras')

# Verify file size
size_mb = os.path.getsize('/content/dermascan.keras') / (1024 * 1024)
print(f'📦 File size: {size_mb:.1f} MB')

In [ ]:
# Download to your local machine
from google.colab import files
print('⬇️  Downloading dermascan.keras...')
files.download('/content/dermascan.keras')
print('\n✅ Done! Place the file at: backend/model/dermascan.keras')

---
## ✅ Next Steps
1. Place `dermascan.keras` in `backend/model/`
2. Copy `backend/.env.example` → `backend/.env` and add your Gemini API key
3. Start the backend: `uvicorn backend.main:app --reload`
4. Open `frontend/index.html` in your browser

Get your free Gemini key at [aistudio.google.com](https://aistudio.google.com)